# Failure Mode 1: Tool Misuse

The agent selects the wrong tool for the task, or passes incorrect arguments to the right tool. This can produce irrelevant or misleading results even when the agent appears to have completed the task.

We evaluate this using three approaches:

| Scorer | Source | Needs expectations? | What it checks |
|---|---|---|---|
| `ToolCallCorrectness` | MLflow native | No | LLM judges tool selection and argument quality |
| `ToolCallCorrectness` (exact match) | MLflow native | Yes | Deterministic name + argument comparison |
| `ToolCorrectness` | DeepEval via MLflow | Yes | Deterministic (by default) tool name matching with optional parameter comparison |

For a detailed explanation of this failure mode and the scorers, see [tool_misuse.md](tool_misuse.md).

### Prerequisites

Start a local MLflow server before running this notebook:

```bash
mlflow server --host 127.0.0.1 --port 5000
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import mlflow
from mlflow.entities import SpanType
from tools import WEATHER_AGENT_TOOLS
from utils import print_eval_results

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("agentic-evaluation")
mlflow.tracing.disable_notebook_display()

EXPERIMENT = mlflow.get_experiment_by_name("agentic-evaluation")

# Clean up old traces for this failure mode
client = mlflow.MlflowClient()
old_traces = mlflow.search_traces(
    experiment_ids=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'tool_misuse'",
    return_type="list",
)
if old_traces:
    client.delete_traces(
        experiment_id=EXPERIMENT.experiment_id,
        trace_ids=[t.info.trace_id for t in old_traces],
    )
    print(f"Cleaned up {len(old_traces)} old traces.")

### Create traces

We create synthetic traces that simulate a weather agent handling the question "What's the weather like in Paris right now?" Three scenarios:

- **Wrong tool:** Agent calls `web_search` instead of `get_weather`
- **Wrong arguments:** Agent calls `get_weather` but passes `"London"` instead of `"Paris"`
- **Correct:** Agent calls `get_weather("Paris")`

In [ ]:
@mlflow.trace(name="weather_agent", span_type=SpanType.AGENT)
def weather_agent_wrong_tool(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, WEATHER_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "tool_misuse", "expected_result": "fail"}
    )

    with mlflow.start_span(name="web_search", span_type=SpanType.TOOL) as span:
        span.set_inputs({"query": "weather in Paris"})
        span.set_outputs({
            "results": [
                {
                    "title": "Paris Travel Guide",
                    "snippet": "Paris is a beautiful city...",
                },
                {
                    "title": "Best time to visit Paris",
                    "snippet": "Spring and fall are ideal...",
                },
            ]
        })

    return (
        "Based on my search, Paris is a beautiful city best visited in spring or fall."
    )


@mlflow.trace(name="weather_agent", span_type=SpanType.AGENT)
def weather_agent_wrong_args(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, WEATHER_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "tool_misuse", "expected_result": "fail"}
    )

    with mlflow.start_span(name="get_weather", span_type=SpanType.TOOL) as span:
        span.set_inputs({"location": "London"})
        span.set_outputs({
            "temperature_celsius": 15,
            "condition": "rainy",
            "humidity": 80,
        })

    return "It's currently 15°C and rainy in London, with 80% humidity."


@mlflow.trace(name="weather_agent", span_type=SpanType.AGENT)
def weather_agent_correct_tool(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, WEATHER_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "tool_misuse", "expected_result": "pass"}
    )

    with mlflow.start_span(name="get_weather", span_type=SpanType.TOOL) as span:
        span.set_inputs({"location": "Paris"})
        span.set_outputs({
            "temperature_celsius": 22,
            "condition": "partly cloudy",
            "humidity": 65,
        })

    return "It's currently 22°C and partly cloudy in Paris, with 65% humidity."


user_msg = [{"role": "user", "content": "What's the weather like in Paris right now?"}]
weather_agent_wrong_tool(user_msg)
weather_agent_wrong_args(user_msg)
weather_agent_correct_tool(user_msg)
mlflow.flush_trace_async_logging()
print("Created 3 traces (2 fail, 1 pass)")

### Load traces

We fetch the Tool Misuse traces we just created:
- **Failing trace (wrong tool):** The agent calls `web_search` instead of `get_weather`
- **Failing trace (wrong args):** The agent calls `get_weather` but passes `"London"` instead of `"Paris"`
- **Passing trace:** The agent correctly calls `get_weather("Paris")`

In [ ]:
tool_misuse_traces = mlflow.search_traces(
    experiment_ids=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'tool_misuse'",
    return_type="list",
)

print(f"Traces found: {len(tool_misuse_traces)}")
for t in tool_misuse_traces:
    tags = t.info.tags or {}
    print(
        f"  [{tags.get('expected_result', '?')}] Request: {str(t.info.request_preview)[:80]}"
    )
    print(f"    Response: {str(t.info.response_preview)[:80]}")
    print()

### Approach 1: Ground-truth-free evaluation (MLflow native — LLM judge)

`ToolCallCorrectness` reads the trace and judges whether the agent's tool selection was appropriate — without needing expected tool calls. No expectations needed, but requires an LLM API key.

In [ ]:
from mlflow.genai.scorers import ToolCallCorrectness

with mlflow.start_run(run_name="tool-misuse-llm-judge") as run:
    results = mlflow.genai.evaluate(
        data=tool_misuse_traces,
        scorers=[ToolCallCorrectness(model="openai:/gpt-4o")],
    )

print_eval_results(results, "tool_call_correctness", EXPERIMENT.experiment_id)

### Approach 2: Deterministic exact match (MLflow native)

`ToolCallCorrectness(should_exact_match=True)` compares tool names and arguments directly against expected tool calls — no LLM needed. Useful for CI/CD pipelines where you want fast, reproducible checks.

To also check that tools were called in a specific order, enable `should_consider_ordering=True`.

We log expected tool calls on the traces using `mlflow.log_expectation()` before evaluation. This runs **after** Approach 1 to avoid affecting the ground-truth-free mode.

In [ ]:
# Log expectations and evaluate in the same cell to keep format consistent.
# ToolCallCorrectness uses "arguments" key for expected tool calls.
for t in tool_misuse_traces:
    mlflow.log_expectation(
        trace_id=t.info.trace_id,
        name="expected_tool_calls",
        value=[{"name": "get_weather", "arguments": {"location": "Paris"}}],
    )

# Re-fetch traces with expectations attached
tool_misuse_traces_with_exp = mlflow.search_traces(
    experiment_ids=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'tool_misuse'",
    return_type="list",
)

with mlflow.start_run(run_name="tool-misuse-exact-match") as run:
    results_exact = mlflow.genai.evaluate(
        data=tool_misuse_traces_with_exp,
        scorers=[ToolCallCorrectness(should_exact_match=True)],
    )

print_eval_results(results_exact, "tool_call_correctness", EXPERIMENT.experiment_id)

### Approach 3: ToolCorrectness (DeepEval — deterministic with expectations)

`ToolCorrectness` compares actual tool calls against expected ones. Two opt-in features:

- **Argument checking:** Off by default (name-only matching). Enable with `evaluation_params=[ToolCallParams.INPUT_PARAMETERS]`.
- **Ordered matching:** Off by default. Enable with `should_consider_ordering=True` to check tool call sequence.

We log expectations using `input_parameters` (the key DeepEval uses, as opposed to `arguments` used by MLflow native).

In [ ]:
from deepeval.test_case import ToolCallParams
from mlflow.genai.scorers.deepeval import ToolCorrectness

# Log expectations and evaluate in the same cell to keep format consistent.
# ToolCorrectness uses "input_parameters" key (different from MLflow's "arguments").
for t in tool_misuse_traces:
    mlflow.log_expectation(
        trace_id=t.info.trace_id,
        name="expected_tool_calls",
        value=[{"name": "get_weather", "input_parameters": {"location": "Paris"}}],
    )

# Re-fetch traces with updated expectations
tool_misuse_traces_deepeval = mlflow.search_traces(
    experiment_ids=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'tool_misuse'",
    return_type="list",
)

# model is passed because ToolCorrectness requires it for initialization,
# but only the deterministic comparison runs since we don't pass available_tools
with mlflow.start_run(run_name="tool-misuse-deepeval") as run:
    results_deepeval = mlflow.genai.evaluate(
        data=tool_misuse_traces_deepeval,
        scorers=[
            ToolCorrectness(
                model="openai:/gpt-4.1-mini",
                evaluation_params=[ToolCallParams.INPUT_PARAMETERS],
            )
        ],
    )

print_eval_results(results_deepeval, "ToolCorrectness", EXPERIMENT.experiment_id)

### Interpreting the results

All three approaches should catch both failure types:
- Wrong tool (`web_search` instead of `get_weather`) → `no`
- Wrong arguments (`get_weather("London")` for a Paris question) → `no`
- Correct usage → `yes`

**When to use which:**
- **LLM judge (Approach 1):** Best for exploratory evaluation — catches both wrong tools and wrong arguments without expectations. No setup needed.
- **Exact match (Approach 2):** Best for CI/CD — deterministic, catches both, no LLM cost. Uses `arguments` key in expectations.
- **ToolCorrectness (Approach 3):** Alternative deterministic check via DeepEval. Distinctive features: argument checking and ordered matching are both opt-in (off by default), and ordering uses LCS-based matching which handles partial matches in multi-step workflows. Uses `input_parameters` key in expectations.

**Note on models:** Approach 1 uses `gpt-4o` and Approach 3 uses `gpt-4.1-mini` because DeepEval 3.9.9 has a JSON parsing issue with `gpt-4o` responses. You can change the `model=` parameter to use any provider MLflow supports.

For full details on each scorer, see [tool_misuse.md](tool_misuse.md).